In [118]:
import pickle
import pandas as pd
import networkx as nx
import ast
from collections import defaultdict
from datetime import datetime

print("Libraries imported successfully!")

Libraries imported successfully!


In [119]:
# Load the graph
graph_path = '../graph/transportation_graph.gpickle'
with open(graph_path, 'rb') as f:
    G = pickle.load(f)

print(f"Graph loaded successfully!")
print(f"Graph type: {type(G)}")
print(f"Total nodes: {G.number_of_nodes()}")
print(f"Total edges: {G.number_of_edges()}")

Graph loaded successfully!
Graph type: <class 'networkx.classes.multidigraph.MultiDiGraph'>
Total nodes: 899
Total edges: 8456


In [120]:
# Load all data files
airports_df = pd.read_csv('../data/final_data/airports_df.csv')
routes_df = pd.read_csv('../data/final_data/routes_df.csv')
flights_df = pd.read_csv('../data/final_data/flights_df.csv')
connections_df = pd.read_csv('../data/final_data/connections.csv')

print("Data files loaded successfully!")
print(f"Airports CSV: {len(airports_df)} rows")
print(f"Routes CSV: {len(routes_df)} rows")
print(f"Flights CSV: {len(flights_df)} rows")
print(f"Connections CSV: {len(connections_df)} rows")

Data files loaded successfully!
Airports CSV: 505 rows
Routes CSV: 3096 rows
Flights CSV: 91290 rows
Connections CSV: 235 rows


In [121]:
# Helper function to parse legs_filtered (same as in build_graph.py)
def parse_legs_filtered(legs_str):
    """Parse the legs_filtered string into a list of tuples."""
    try:
        legs = ast.literal_eval(legs_str)
        result = []
        for leg in legs:
            if len(leg) >= 2:
                station_info = leg[0]  # [station_name, country]
                time_info = leg[1]    # [departure_time, arrival_time]
                if len(station_info) >= 2 and len(time_info) >= 2:
                    station_name = station_info[0]
                    country = station_info[1]
                    result.append((station_name, country))
        return result
    except Exception as e:
        return []

print("Helper function defined!")

Helper function defined!


In [122]:
# ============================================================================
# VALIDATION 1: Check Airport Nodes
# ============================================================================
print("="*70)
print("VALIDATION 1: Airport Nodes")
print("="*70)

# Filter flights for 2026-01-21 and 2026-01-22 (same as in build_graph.py)
flights_df_filtered = flights_df.copy()
flights_df_filtered['departure_scheduled'] = pd.to_datetime(
    flights_df_filtered['departure_scheduled'], errors='coerce'
)
flights_df_filtered = flights_df_filtered[
    (flights_df_filtered['departure_scheduled'].dt.date == pd.Timestamp('2026-01-21').date()) |
    (flights_df_filtered['departure_scheduled'].dt.date == pd.Timestamp('2026-01-22').date())
]

# Extract unique airports from filtered flights (same logic as build_graph.py)
unique_airports_from_flights = set()
for _, row in flights_df_filtered.iterrows():
    dep_airport = row['departure_airport']
    arr_airport = row['arrival_airport']
    if pd.notna(dep_airport):
        unique_airports_from_flights.add(dep_airport)
    if pd.notna(arr_airport):
        unique_airports_from_flights.add(arr_airport)

airport_count_from_flights = len(unique_airports_from_flights)

# Count airports in graph
airport_nodes_graph = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'airport']
airport_count_graph = len(airport_nodes_graph)

print(f"Unique airports in filtered flights (2026-01-21 and 2026-01-22): {airport_count_from_flights}")
print(f"Airport nodes in graph: {airport_count_graph}")

if airport_count_from_flights == airport_count_graph:
    print("✓ PASS: Airport counts match!")
else:
    print("✗ FAIL: Airport counts do not match!")
    
# Check if all airports from flights are in graph
airports_in_graph_set = set(airport_nodes_graph)
missing_in_graph = unique_airports_from_flights - airports_in_graph_set
extra_in_graph = airports_in_graph_set - unique_airports_from_flights

if missing_in_graph:
    print(f"  ⚠ Missing in graph: {len(missing_in_graph)} airports")
    print(f"    Examples: {list(missing_in_graph)[:5]}")
if extra_in_graph:
    print(f"  ⚠ Extra in graph: {len(extra_in_graph)} airports")
    print(f"    Examples: {list(extra_in_graph)[:5]}")
if not missing_in_graph and not extra_in_graph:
    print("✓ PASS: All airports match exactly!")

VALIDATION 1: Airport Nodes
Unique airports in filtered flights (2026-01-21 and 2026-01-22): 475
Airport nodes in graph: 475
✓ PASS: Airport counts match!
✓ PASS: All airports match exactly!


In [123]:
# ============================================================================
# VALIDATION 2: Check Station Nodes
# ============================================================================
print("\n" + "="*70)
print("VALIDATION 2: Station Nodes")
print("="*70)

# Extract unique stations from routes_df.csv
stations_in_csv = set()
for _, row in routes_df.iterrows():
    legs_filtered = row['legs_filtered']
    if pd.notna(legs_filtered):
        legs = parse_legs_filtered(legs_filtered)
        for station_name, country in legs:
            stations_in_csv.add(station_name)

station_count_csv = len(stations_in_csv)

# Count stations in graph
station_nodes_graph = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'station']
station_count_graph = len(station_nodes_graph)

print(f"Unique stations in CSV: {station_count_csv}")
print(f"Station nodes in graph: {station_count_graph}")

if station_count_csv == station_count_graph:
    print("✓ PASS: Station counts match!")
else:
    print("✗ FAIL: Station counts do not match!")
    
# Check if all stations in CSV are in graph
stations_in_graph_set = set(station_nodes_graph)
missing_in_graph = stations_in_csv - stations_in_graph_set
extra_in_graph = stations_in_graph_set - stations_in_csv

if missing_in_graph:
    print(f"  ⚠ Missing in graph: {len(missing_in_graph)} stations")
    print(f"    Examples: {list(missing_in_graph)[:5]}")
if extra_in_graph:
    print(f"  ⚠ Extra in graph: {len(extra_in_graph)} stations")
    print(f"    Examples: {list(extra_in_graph)[:5]}")
if not missing_in_graph and not extra_in_graph:
    print("✓ PASS: All stations match exactly!")


VALIDATION 2: Station Nodes
Unique stations in CSV: 424
Station nodes in graph: 424
✓ PASS: Station counts match!
✓ PASS: All stations match exactly!


In [124]:
# ============================================================================
# VALIDATION 3: Check Flight Edges (for 2026-01-21 and 2026-01-22)
# ============================================================================
print("\n" + "="*70)
print("VALIDATION 3: Flight Edges (2026-01-21 and 2026-01-22)")
print("="*70)

# Filter flights for 2026-01-21 and 2026-01-22 (same as in build_graph.py)
flights_df_filtered = flights_df.copy()
flights_df_filtered['departure_scheduled'] = pd.to_datetime(
    flights_df_filtered['departure_scheduled'], errors='coerce'
)
flights_df_filtered = flights_df_filtered[
    (flights_df_filtered['departure_scheduled'].dt.date == pd.Timestamp('2026-01-21').date()) |
    (flights_df_filtered['departure_scheduled'].dt.date == pd.Timestamp('2026-01-22').date())
]

print(f"Total flights in CSV (filtered): {len(flights_df_filtered)}")

# Count unique flight routes in CSV (same logic as build_graph.py)
flight_edges_csv = set()
total_flights_count = 0  # Total number of individual flights
for _, row in flights_df_filtered.iterrows():
    dep_airport = row['departure_airport']
    arr_airport = row['arrival_airport']
    dep_time_utc = row['departure_scheduled_absolute_utc']
    arr_time_utc = row['arrival_scheduled_absolute_utc']
    
    if pd.notna(dep_airport) and pd.notna(arr_airport) and pd.notna(dep_time_utc) and pd.notna(arr_time_utc):
        flight_edges_csv.add((dep_airport, arr_airport))
        total_flights_count += 1

flight_edge_count_csv = len(flight_edges_csv)

# Count flight edges in graph
flight_edges_graph = [(u, v, d) for u, v, d in G.edges(data=True) if d.get('edge_type') == 'flight']
flight_edge_count_graph = len(flight_edges_graph)

# Count total time pairs (individual flights) in graph
total_time_pairs_graph = 0
for u, v, d in flight_edges_graph:
    time_pairs = d.get('time_pairs', [])
    if time_pairs:
        total_time_pairs_graph += len(time_pairs)

print(f"Unique flight routes in CSV: {flight_edge_count_csv}")
print(f"Flight edges in graph: {flight_edge_count_graph}")

if flight_edge_count_csv == flight_edge_count_graph:
    print("✓ PASS: Flight edge counts match!")
else:
    print("✗ FAIL: Flight edge counts do not match!")
    
# Check if all flight routes in CSV are in graph
flight_edges_graph_set = set((u, v) for u, v, d in flight_edges_graph)
missing_in_graph = flight_edges_csv - flight_edges_graph_set
extra_in_graph = flight_edges_graph_set - flight_edges_csv

if missing_in_graph:
    print(f"  ⚠ Missing in graph: {len(missing_in_graph)} flight routes")
    print(f"    Examples: {list(missing_in_graph)[:5]}")
if extra_in_graph:
    print(f"  ⚠ Extra in graph: {len(extra_in_graph)} flight routes")
    print(f"    Examples: {list(extra_in_graph)[:5]}")
if not missing_in_graph and not extra_in_graph:
    print("✓ PASS: All flight routes match exactly!")

# Check total number of flights (time pairs)
print(f"\nTotal individual flights in CSV (with valid data): {total_flights_count}")
print(f"Total time pairs in graph flight edges: {total_time_pairs_graph}")

if total_flights_count == total_time_pairs_graph:
    print("✓ PASS: Total flight counts match!")
else:
    print("✗ FAIL: Total flight counts do not match!")
    print(f"  Difference: {abs(total_flights_count - total_time_pairs_graph)}")
    
# Also check that airports in flights exist in graph
flights_airports = set(flights_df_filtered['departure_airport'].dropna()) | \
                  set(flights_df_filtered['arrival_airport'].dropna())
missing_airports = flights_airports - airports_in_graph_set
if missing_airports:
    print(f"  ⚠ WARNING: {len(missing_airports)} airports in flights are not in graph")
    print(f"    Examples: {list(missing_airports)[:5]}")
else:
    print("✓ PASS: All airports in flights exist in graph!")

# ============================================================================
# VALIDATION 3b: Check Time Shifts for Flights on 2026-01-22
# ============================================================================
print(f"\n{'='*70}")
print("VALIDATION 3b: Time Shifts for Flights on 2026-01-22")
print(f"{'='*70}")

# Separate flights by date
flights_jan21 = flights_df_filtered[
    flights_df_filtered['departure_scheduled'].dt.date == pd.Timestamp('2026-01-21').date()
]
flights_jan22 = flights_df_filtered[
    flights_df_filtered['departure_scheduled'].dt.date == pd.Timestamp('2026-01-22').date()
]

print(f"Flights on 2026-01-21: {len(flights_jan21)}")
print(f"Flights on 2026-01-22: {len(flights_jan22)}")

# Check that flights on 2026-01-22 have times shifted by 1440 minutes
shift_errors = []
shift_correct = 0

for _, row in flights_jan22.iterrows():
    dep_airport = row['departure_airport']
    arr_airport = row['arrival_airport']
    original_dep_time = row['departure_scheduled_absolute_utc']
    original_arr_time = row['arrival_scheduled_absolute_utc']
    
    if pd.notna(dep_airport) and pd.notna(arr_airport) and pd.notna(original_dep_time) and pd.notna(original_arr_time):
        expected_dep_time = int(original_dep_time) + 1440
        expected_arr_time = int(original_arr_time) + 1440
        
        # Find the edge in graph
        edge_found = False
        for u, v, d in flight_edges_graph:
            if u == dep_airport and v == arr_airport:
                time_pairs = d.get('time_pairs', [])
                # Check if the shifted time pair exists
                if (expected_dep_time, expected_arr_time) in time_pairs:
                    shift_correct += 1
                    edge_found = True
                    break
        
        if not edge_found:
            shift_errors.append({
                'route': (dep_airport, arr_airport),
                'original': (int(original_dep_time), int(original_arr_time)),
                'expected': (expected_dep_time, expected_arr_time)
            })

if shift_errors:
    print(f"✗ FAIL: {len(shift_errors)} flights on 2026-01-22 have incorrect time shifts")
    print(f"  Examples:")
    for err in shift_errors[:5]:
        print(f"    Route {err['route']}: original {err['original']}, expected {err['expected']}")
else:
    print(f"✓ PASS: All {shift_correct} flights on 2026-01-22 have correct time shifts (+1440 minutes)")

# Also verify that flights on 2026-01-21 are NOT shifted
no_shift_errors = []
no_shift_correct = 0

for _, row in flights_jan21.iterrows():
    dep_airport = row['departure_airport']
    arr_airport = row['arrival_airport']
    original_dep_time = row['departure_scheduled_absolute_utc']
    original_arr_time = row['arrival_scheduled_absolute_utc']
    
    if pd.notna(dep_airport) and pd.notna(arr_airport) and pd.notna(original_dep_time) and pd.notna(original_arr_time):
        expected_dep_time = int(original_dep_time)
        expected_arr_time = int(original_arr_time)
        
        # Find the edge in graph
        edge_found = False
        for u, v, d in flight_edges_graph:
            if u == dep_airport and v == arr_airport:
                time_pairs = d.get('time_pairs', [])
                # Check if the original (unshifted) time pair exists
                if (expected_dep_time, expected_arr_time) in time_pairs:
                    no_shift_correct += 1
                    edge_found = True
                    break
        
        if not edge_found:
            no_shift_errors.append({
                'route': (dep_airport, arr_airport),
                'expected': (expected_dep_time, expected_arr_time)
            })

if no_shift_errors:
    print(f"✗ FAIL: {len(no_shift_errors)} flights on 2026-01-21 are missing or incorrectly shifted")
    print(f"  Examples:")
    for err in no_shift_errors[:5]:
        print(f"    Route {err['route']}: expected {err['expected']}")
else:
    print(f"✓ PASS: All {no_shift_correct} flights on 2026-01-21 have correct (unshifted) times")


VALIDATION 3: Flight Edges (2026-01-21 and 2026-01-22)
Total flights in CSV (filtered): 25084
Unique flight routes in CSV: 6558
Flight edges in graph: 6558
✓ PASS: Flight edge counts match!
✓ PASS: All flight routes match exactly!

Total individual flights in CSV (with valid data): 25084
Total time pairs in graph flight edges: 25084
✓ PASS: Total flight counts match!
✓ PASS: All airports in flights exist in graph!

VALIDATION 3b: Time Shifts for Flights on 2026-01-22
Flights on 2026-01-21: 11723
Flights on 2026-01-22: 13361
✓ PASS: All 13361 flights on 2026-01-22 have correct time shifts (+1440 minutes)
✓ PASS: All 11723 flights on 2026-01-21 have correct (unshifted) times


In [125]:
# ============================================================================
# VALIDATION 4: Check Train Edges
# ============================================================================
print("\n" + "="*70)
print("VALIDATION 4: Train Edges")
print("="*70)

# Helper function to parse legs_filtered_absolute_utc (same as in build_graph.py)
def parse_legs_filtered_absolute_utc(legs_str):
    """Parse the legs_filtered_absolute_utc string into a list of tuples."""
    try:
        legs = ast.literal_eval(legs_str)
        result = []
        for leg in legs:
            if len(leg) >= 2:
                station_info = leg[0]  # [station_name, country]
                time_info = leg[1]     # [departure_time_utc, arrival_time_utc]
                if len(station_info) >= 2 and len(time_info) >= 2:
                    station_name = station_info[0]
                    country = station_info[1]
                    dep_time_utc = time_info[0]
                    arr_time_utc = time_info[1]
                    if isinstance(dep_time_utc, str):
                        dep_time_utc = int(dep_time_utc) if dep_time_utc.isdigit() else None
                    if isinstance(arr_time_utc, str):
                        arr_time_utc = int(arr_time_utc) if arr_time_utc.isdigit() else None
                    result.append((station_name, country, dep_time_utc, arr_time_utc))
        return result
    except Exception as e:
        return []

# Count unique train routes in CSV (same logic as build_graph.py)
train_edges_csv = set()
total_train_segments_count = 0  # Total number of individual train route segments
for _, row in routes_df.iterrows():
    legs_utc = row['legs_filtered_absolute_utc']
    if pd.notna(legs_utc):
        legs = parse_legs_filtered_absolute_utc(legs_utc)
        
        # Create edges between consecutive stops
        for i in range(len(legs) - 1):
            station1, country1, dep_time1, arr_time1 = legs[i]
            station2, country2, dep_time2, arr_time2 = legs[i + 1]
            
            # According to the spec: use arr_time1 and dep_time2
            if arr_time1 is not None and dep_time2 is not None:
                train_edges_csv.add((station1, station2))
                total_train_segments_count += 1

train_edge_count_csv = len(train_edges_csv)

# Count train edges in graph
train_edges_graph = [(u, v, d) for u, v, d in G.edges(data=True) if d.get('edge_type') == 'train']
train_edge_count_graph = len(train_edges_graph)

# Count total time pairs (individual train segments) in graph
total_time_pairs_graph = 0
for u, v, d in train_edges_graph:
    time_pairs = d.get('time_pairs', [])
    if time_pairs:
        total_time_pairs_graph += len(time_pairs)

print(f"Unique train routes in CSV: {train_edge_count_csv}")
print(f"Train edges in graph: {train_edge_count_graph}")

if train_edge_count_csv == train_edge_count_graph:
    print("✓ PASS: Train edge counts match!")
else:
    print("✗ FAIL: Train edge counts do not match!")
    
# Check if all train routes in CSV are in graph
train_edges_graph_set = set((u, v) for u, v, d in train_edges_graph)
missing_in_graph = train_edges_csv - train_edges_graph_set
extra_in_graph = train_edges_graph_set - train_edges_csv

if missing_in_graph:
    print(f"  ⚠ Missing in graph: {len(missing_in_graph)} train routes")
    print(f"    Examples: {list(missing_in_graph)[:5]}")
if extra_in_graph:
    print(f"  ⚠ Extra in graph: {len(extra_in_graph)} train routes")
    print(f"    Examples: {list(extra_in_graph)[:5]}")
if not missing_in_graph and not extra_in_graph:
    print("✓ PASS: All train routes match exactly!")

# Check total number of train route segments (time pairs)
print(f"\nTotal individual train route segments in CSV: {total_train_segments_count}")
print(f"Total time pairs in graph train edges: {total_time_pairs_graph}")
print(f"Expected (original + duplicated with +1440): {total_train_segments_count * 2}")

if total_time_pairs_graph == total_train_segments_count * 2:
    print("✓ PASS: Total train segment counts match (duplicated correctly)!")
else:
    print("✗ FAIL: Total train segment counts do not match!")
    print(f"  Difference: {abs(total_train_segments_count * 2 - total_time_pairs_graph)}")

# ============================================================================
# VALIDATION 4b: Check Time Shifts for Train Routes (Duplication with +1440)
# ============================================================================
print(f"\n{'='*70}")
print("VALIDATION 4b: Time Shifts for Train Routes (Duplication)")
print(f"{'='*70}")

# Verify that each train route has duplicated time pairs (original + shifted by 1440)
duplication_errors = []
duplication_correct = 0

for _, row in routes_df.iterrows():
    legs_utc = row['legs_filtered_absolute_utc']
    if pd.notna(legs_utc):
        legs = parse_legs_filtered_absolute_utc(legs_utc)
        
        # Create edges between consecutive stops
        for i in range(len(legs) - 1):
            station1, country1, dep_time1, arr_time1 = legs[i]
            station2, country2, dep_time2, arr_time2 = legs[i + 1]
            
            if arr_time1 is not None and dep_time2 is not None:
                original_time_pair = (int(arr_time1), int(dep_time2))
                expected_shifted_pair = (int(arr_time1) + 1440, int(dep_time2) + 1440)
                
                # Find the edge in graph
                edge_found = False
                for u, v, d in train_edges_graph:
                    if u == station1 and v == station2:
                        time_pairs = d.get('time_pairs', [])
                        # Check if both original and shifted pairs exist
                        has_original = original_time_pair in time_pairs
                        has_shifted = expected_shifted_pair in time_pairs
                        
                        if has_original and has_shifted:
                            duplication_correct += 1
                            edge_found = True
                        elif not has_original or not has_shifted:
                            duplication_errors.append({
                                'route': (station1, station2),
                                'original': original_time_pair,
                                'expected_shifted': expected_shifted_pair,
                                'has_original': has_original,
                                'has_shifted': has_shifted,
                                'time_pairs_in_graph': time_pairs[:3]  # Show first 3 for debugging
                            })
                        break

if duplication_errors:
    print(f"✗ FAIL: {len(duplication_errors)} train route segments have incorrect duplication")
    print(f"  Examples:")
    for err in duplication_errors[:5]:
        print(f"    Route {err['route']}:")
        print(f"      Original: {err['original']} (found: {err['has_original']})")
        print(f"      Shifted: {err['expected_shifted']} (found: {err['has_shifted']})")
        print(f"      Graph time pairs (first 3): {err['time_pairs_in_graph']}")
else:
    print(f"✓ PASS: All {duplication_correct} train route segments have correct duplication (original + shifted by +1440)")


VALIDATION 4: Train Edges
Unique train routes in CSV: 1450
Train edges in graph: 1450
✓ PASS: Train edge counts match!
✓ PASS: All train routes match exactly!

Total individual train route segments in CSV: 12449
Total time pairs in graph train edges: 24898
Expected (original + duplicated with +1440): 24898
✓ PASS: Total train segment counts match (duplicated correctly)!

VALIDATION 4b: Time Shifts for Train Routes (Duplication)
✓ PASS: All 12449 train route segments have correct duplication (original + shifted by +1440)


In [126]:
# ============================================================================
# VALIDATION 5: Check Connection Edges (bidirectional)
# ============================================================================
print("\n" + "="*70)
print("VALIDATION 5: Connection Edges")
print("="*70)

# Count connection edges in CSV (should be 2x since bidirectional)
connection_rows_csv = connections_df[
    connections_df['Origin'].notna() & 
    connections_df['Destination'].notna() & 
    connections_df['connection_time'].notna()
]
connection_count_csv = len(connection_rows_csv)
# Since connections are bidirectional, we expect 2x edges in graph
expected_connection_edges = connection_count_csv * 2

# Count connection edges in graph
connection_edges_graph = [(u, v) for u, v, d in G.edges(data=True) if d.get('edge_type') == 'connection']
connection_edge_count_graph = len(connection_edges_graph)

print(f"Connection rows in CSV: {connection_count_csv}")
print(f"Expected connection edges in graph (2x bidirectional): {expected_connection_edges}")
print(f"Connection edges in graph: {connection_edge_count_graph}")

if expected_connection_edges == connection_edge_count_graph:
    print("✓ PASS: Connection edge counts match!")
else:
    print("✗ FAIL: Connection edge counts do not match!")
    print(f"  Difference: {abs(expected_connection_edges - connection_edge_count_graph)}")
    
# Check if all connections in CSV are in graph (both directions)
connection_edges_expected = set()
connection_rows_list = []
for _, row in connection_rows_csv.iterrows():
    origin = row['Origin']
    destination = row['Destination']
    connection_edges_expected.add((origin, destination))
    connection_edges_expected.add((destination, origin))  # bidirectional
    connection_rows_list.append((origin, destination))

connection_edges_graph_set = set(connection_edges_graph)
missing_in_graph = connection_edges_expected - connection_edges_graph_set
extra_in_graph = connection_edges_graph_set - connection_edges_expected

if missing_in_graph:
    print(f"  ⚠ Missing in graph: {len(missing_in_graph)} connection edges")
    print(f"    Examples: {list(missing_in_graph)[:5]}")
if extra_in_graph:
    print(f"  ⚠ Extra in graph: {len(extra_in_graph)} connection edges")
    print(f"    Examples: {list(extra_in_graph)[:5]}")
if not missing_in_graph and not extra_in_graph:
    print("✓ PASS: All connection edges match exactly!")
    
# Detailed analysis of missing connections
all_nodes_in_graph = set(G.nodes())
airport_nodes_in_graph = set(n for n, d in G.nodes(data=True) if d.get('node_type') == 'airport')
station_nodes_in_graph = set(n for n, d in G.nodes(data=True) if d.get('node_type') == 'station')

# Analyze which connections are missing and why
missing_connections_analysis = []
for origin, destination in connection_rows_list:
    origin_in_graph = origin in all_nodes_in_graph
    dest_in_graph = destination in all_nodes_in_graph
    
    # Check if both directions are missing
    forward_missing = (origin, destination) not in connection_edges_graph_set
    backward_missing = (destination, origin) not in connection_edges_graph_set
    
    if not origin_in_graph or not dest_in_graph or forward_missing or backward_missing:
        origin_type = "airport" if origin in airport_nodes_in_graph else ("station" if origin in station_nodes_in_graph else "MISSING")
        dest_type = "airport" if destination in airport_nodes_in_graph else ("station" if destination in station_nodes_in_graph else "MISSING")
        
        missing_connections_analysis.append({
            'origin': origin,
            'destination': destination,
            'origin_in_graph': origin_in_graph,
            'dest_in_graph': dest_in_graph,
            'origin_type': origin_type,
            'dest_type': dest_type,
            'forward_missing': forward_missing,
            'backward_missing': backward_missing
        })

if missing_connections_analysis:
    print(f"\n{'='*70}")
    print("DETAILED ANALYSIS OF MISSING CONNECTIONS")
    print(f"{'='*70}")
    
    # Group by reason
    both_nodes_missing = [c for c in missing_connections_analysis if not c['origin_in_graph'] and not c['dest_in_graph']]
    origin_missing = [c for c in missing_connections_analysis if not c['origin_in_graph'] and c['dest_in_graph']]
    dest_missing = [c for c in missing_connections_analysis if c['origin_in_graph'] and not c['dest_in_graph']]
    both_present_but_missing = [c for c in missing_connections_analysis if c['origin_in_graph'] and c['dest_in_graph']]
    
    print(f"\nSummary:")
    print(f"  Connections with both nodes missing: {len(both_nodes_missing)}")
    print(f"  Connections with origin missing: {len(origin_missing)}")
    print(f"  Connections with destination missing: {len(dest_missing)}")
    print(f"  Connections with both nodes present but edges missing: {len(both_present_but_missing)}")
    
    if both_nodes_missing:
        print(f"\n  Both nodes missing ({len(both_nodes_missing)} connections):")
        for c in both_nodes_missing[:10]:
            print(f"    {c['origin']} ({c['origin_type']}) <-> {c['destination']} ({c['dest_type']})")
    
    if origin_missing:
        print(f"\n  Origin missing ({len(origin_missing)} connections):")
        for c in origin_missing[:10]:
            print(f"    {c['origin']} ({c['origin_type']}) <-> {c['destination']} ({c['dest_type']})")
    
    if dest_missing:
        print(f"\n  Destination missing ({len(dest_missing)} connections):")
        for c in dest_missing[:10]:
            print(f"    {c['origin']} ({c['origin_type']}) <-> {c['destination']} ({c['dest_type']})")
    
    if both_present_but_missing:
        print(f"\n  ⚠ UNEXPECTED: Both nodes present but edges missing ({len(both_present_but_missing)} connections):")
        for c in both_present_but_missing:
            print(f"    {c['origin']} ({c['origin_type']}) <-> {c['destination']} ({c['dest_type']})")
            print(f"      Forward missing: {c['forward_missing']}, Backward missing: {c['backward_missing']}")
    
    # Check if missing airports are in the airports CSV but not in filtered flights
    missing_airports = set()
    for c in missing_connections_analysis:
        if not c['origin_in_graph'] and c['origin_type'] == 'MISSING':
            # Check if it's an airport code (3 letters, uppercase)
            if len(c['origin']) == 3 and c['origin'].isupper():
                missing_airports.add(c['origin'])
        if not c['dest_in_graph'] and c['dest_type'] == 'MISSING':
            if len(c['destination']) == 3 and c['destination'].isupper():
                missing_airports.add(c['destination'])
    
    if missing_airports:
        print(f"\n  Missing airports (likely not in filtered flights): {len(missing_airports)}")
        print(f"    {sorted(list(missing_airports))}")
        
        # Check if they exist in airports_df
        if 'airports_df' in locals():
            missing_in_airports_csv = missing_airports - set(airports_df['iata_code'].dropna())
            if missing_in_airports_csv:
                print(f"    ⚠ Note: {len(missing_in_airports_csv)} of these are not even in airports_df.csv: {sorted(list(missing_in_airports_csv))}")
    
    print(f"\n{'='*70}")
    
# Check that connection nodes exist in graph
connection_nodes = set(connection_rows_csv['Origin'].dropna()) | \
                  set(connection_rows_csv['Destination'].dropna())
missing_nodes = connection_nodes - all_nodes_in_graph
if missing_nodes:
    print(f"  ⚠ WARNING: {len(missing_nodes)} connection nodes are not in graph")
    print(f"    Examples: {list(missing_nodes)[:5]}")
    
    # Categorize missing nodes
    missing_airports = [n for n in missing_nodes if len(n) == 3 and n.isupper()]
    missing_stations = [n for n in missing_nodes if n not in missing_airports]
    
    if missing_airports:
        print(f"    Missing airports: {len(missing_airports)} - {sorted(missing_airports)}")
    if missing_stations:
        print(f"    Missing stations: {len(missing_stations)} - {sorted(missing_stations)[:10]}")
else:
    print("✓ PASS: All connection nodes exist in graph!")


VALIDATION 5: Connection Edges
Connection rows in CSV: 235
Expected connection edges in graph (2x bidirectional): 470
Connection edges in graph: 448
✗ FAIL: Connection edge counts do not match!
  Difference: 22
  ⚠ Missing in graph: 22 connection edges
    Examples: [('KBP', 'Kiev Pass'), ('IEV', 'Kiev Pass'), ('DLE', 'Dole Ville'), ('UDJ', 'Chop'), ('IEV', 'KBP')]

DETAILED ANALYSIS OF MISSING CONNECTIONS

Summary:
  Connections with both nodes missing: 1
  Connections with origin missing: 9
  Connections with destination missing: 1
  Connections with both nodes present but edges missing: 0

  Both nodes missing (1 connections):
    KBP (MISSING) <-> IEV (MISSING)

  Origin missing (9 connections):
    ODE (MISSING) <-> Odense st (station)
    ETZ (MISSING) <-> Lorraine TGV (station)
    DIJ (MISSING) <-> Dijon Ville (station)
    DLE (MISSING) <-> Dole Ville (station)
    ARW (MISSING) <-> Arad (station)
    KBP (MISSING) <-> Kiev Pass (station)
    IEV (MISSING) <-> Kiev Pass (stat

In [127]:
# ============================================================================
# VALIDATION 6: Summary and Overall Check
# ============================================================================
print("\n" + "="*70)
print("VALIDATION 6: Summary")
print("="*70)

# Count nodes by type
airport_nodes_count = sum(1 for n, d in G.nodes(data=True) if d.get('node_type') == 'airport')
station_nodes_count = sum(1 for n, d in G.nodes(data=True) if d.get('node_type') == 'station')
total_nodes = G.number_of_nodes()

# Count edges by type
flight_edges_count = sum(1 for u, v, d in G.edges(data=True) if d.get('edge_type') == 'flight')
train_edges_count = sum(1 for u, v, d in G.edges(data=True) if d.get('edge_type') == 'train')
connection_edges_count = sum(1 for u, v, d in G.edges(data=True) if d.get('edge_type') == 'connection')
total_edges = G.number_of_edges()

# Count total time pairs in graph
total_flight_time_pairs_graph = 0
for u, v, d in G.edges(data=True):
    if d.get('edge_type') == 'flight':
        time_pairs = d.get('time_pairs', [])
        if time_pairs:
            total_flight_time_pairs_graph += len(time_pairs)

total_train_time_pairs_graph = 0
for u, v, d in G.edges(data=True):
    if d.get('edge_type') == 'train':
        time_pairs = d.get('time_pairs', [])
        if time_pairs:
            total_train_time_pairs_graph += len(time_pairs)

print(f"\nGraph Summary:")
print(f"  Total nodes: {total_nodes}")
print(f"    - Airport nodes: {airport_nodes_count}")
print(f"    - Station nodes: {station_nodes_count}")
print(f"  Total edges: {total_edges}")
print(f"    - Flight edges: {flight_edges_count}")
print(f"    - Train edges: {train_edges_count}")
print(f"    - Connection edges: {connection_edges_count}")
print(f"  Total time pairs:")
print(f"    - Flight time pairs: {total_flight_time_pairs_graph}")
print(f"    - Train time pairs: {total_train_time_pairs_graph}")

print(f"\nData Summary:")
print(f"  Airports from filtered flights (2026-01-21 and 2026-01-22): {airport_count_from_flights}")
print(f"  Stations in CSV: {station_count_csv}")
print(f"  Flight routes in CSV (filtered): {flight_edge_count_csv}")
print(f"  Total flights in CSV (filtered): {total_flights_count}")
print(f"    Note: Flights on 2026-01-22 are shifted by +1440 minutes")
print(f"  Train routes in CSV: {train_edge_count_csv}")
print(f"  Total train segments in CSV: {total_train_segments_count}")
print(f"  Expected train time pairs (duplicated): {total_train_segments_count * 2}")
print(f"  Connection rows in CSV: {connection_count_csv}")

# Overall validation result
all_valid = (
    airport_count_from_flights == airport_count_graph and
    station_count_csv == station_count_graph and
    flight_edge_count_csv == flight_edge_count_graph and
    total_flights_count == total_flight_time_pairs_graph and
    train_edge_count_csv == train_edge_count_graph and
    total_train_segments_count * 2 == total_train_time_pairs_graph and
    expected_connection_edges == connection_edge_count_graph
)

print(f"\n{'='*70}")
if all_valid:
    print("✓✓✓ ALL VALIDATIONS PASSED! ✓✓✓")
    print("The graph accurately represents the data!")
else:
    print("✗✗✗ SOME VALIDATIONS FAILED ✗✗✗")
    print("Please review the details above.")
print(f"{'='*70}")


VALIDATION 6: Summary

Graph Summary:
  Total nodes: 899
    - Airport nodes: 475
    - Station nodes: 424
  Total edges: 8456
    - Flight edges: 6558
    - Train edges: 1450
    - Connection edges: 448
  Total time pairs:
    - Flight time pairs: 25084
    - Train time pairs: 24898

Data Summary:
  Airports from filtered flights (2026-01-21 and 2026-01-22): 475
  Stations in CSV: 424
  Flight routes in CSV (filtered): 6558
  Total flights in CSV (filtered): 25084
    Note: Flights on 2026-01-22 are shifted by +1440 minutes
  Train routes in CSV: 1450
  Total train segments in CSV: 12449
  Expected train time pairs (duplicated): 24898
  Connection rows in CSV: 235

✗✗✗ SOME VALIDATIONS FAILED ✗✗✗
Please review the details above.
